# Section 4: exploring the GEANT4 output

Problem Set 3 only needs the total energy each muon deposits. The ROOT file holds much more: every individual energy deposit, with its position. Here we open the whole file and look around.

**Get data first** (in `Phys245-PS3/build`, with `conda activate g4env`; the macros are in this folder, so copy them to `build` or give their path). The output file name is fixed, so rename it after each run:
```
./examplePS3 mu_10GeV_1event.mac && mv output_default.root single_mu_10GeV.root
./examplePS3 mu_10TeV_1event.mac && mv output_default.root single_mu_10TeV.root
./examplePS3 run.mac             && mv output_default.root muons_10GeV.root    # 1000 muons of 10 GeV
```
To run this notebook in the same environment: `conda install jupyterlab` (once), then `jupyter lab`.

In [ ]:
import os
import numpy as np
import uproot
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

plt.rcParams.update({
    "font.family": "serif", "mathtext.fontset": "cm",
    "axes.labelsize": 15, "xtick.labelsize": 12, "ytick.labelsize": 12,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.major.size": 7, "ytick.major.size": 7, "xtick.minor.size": 4, "ytick.minor.size": 4,
    "lines.linewidth": 2.0, "legend.frameon": False, "legend.fontsize": 11,
    "figure.figsize": (6, 4), "figure.dpi": 200,
})

FILE = "muons_10GeV.root"   # your renamed output file
E0 = 10.0                   # beam kinetic energy [GeV] (/gun/energy)
L = 1000.0                  # detector length along the beam [cm]
MIP = 2.105                 # minimum-ionizing dE/dx in liquid argon [MeV/cm] (PDG)

## 0. One muon at 10 GeV vs. one at 10 TeV

First, where do these two energies sit on the muon energy-loss curve? (PDG, muons in copper; argon has the same shape.)

![Muon stopping power vs. momentum (PDG)](bb.png)

A 10 GeV muon sits just above minimum ionization, where it loses energy almost entirely by ionization (Bethe–Bloch). A 10 TeV muon is past the muon critical energy $E_{\mu c}$ ($\approx 485$ GeV in argon), where radiative losses dominate.

Now one muon of each: where along the detector does the energy go?

In [ ]:
single = {"10 GeV": "single_mu_10GeV.root", "10 TeV": "single_mu_10TeV.root"}
zb = np.linspace(0, L, 101)                                          # 10 cm slices

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
for (label, fname), color in zip(single.items(), ["#4C72B0", "#DD8452"]):
    s = uproot.open(fname)["showerEDep"].arrays(["E", "z"], library="np")
    Es, zs = s["E"] * 1e3, s["z"]                                    # [MeV], [cm]
    ax1.stairs(np.histogram(zs, bins=zb, weights=Es)[0] / np.diff(zb), zb, color=color, lw=1.5, label=label)
    order = np.argsort(zs)
    ax2.plot(zs[order], np.cumsum(Es[order]) / 1e3, color=color, label=f"{label}: {Es.sum()/1e3:.1f} GeV total")
ax1.axhline(MIP, color="gray", ls=":", label="MIP")
ax1.set_yscale("log")
ax1.set_xlabel("Depth $z$ [cm]")
ax1.set_ylabel("$dE/dz$ [MeV/cm]")
ax1.legend(fontsize=10)
ax2.plot([0, L], [0, MIP * L / 1e3], color="gray", ls=":", label="MIP")
ax2.set_xlabel("Depth $z$ [cm]")
ax2.set_ylabel("Energy deposited so far [GeV]")
ax2.legend(fontsize=10)
plt.tight_layout()
plt.show()

The **10 GeV** muon leaves a steady trail near the MIP rate: its losses are ionization, many small frequent collisions, so the cumulative curve is nearly a straight line. The **10 TeV** muon has the same trail plus occasional big jumps: a radiated photon or $e^+e^-$ pair that starts an electromagnetic shower. Above the muon critical energy ($\approx 485$ GeV in argon) these rare, **stochastic** radiative losses dominate.

Each macro gives the same muon every time (the random seed is fixed; the 10 GeV macro sets one that gives a typical muon). Change or add `/random/setSeeds 1 2` (any two numbers) at the top of a macro to see a different muon; at 10 TeV the total varies enormously from muon to muon.

## 1. What's in the file?

Two trees (tables):

- `edep`: one row per **event**. `Edep` = total energy deposited in the argon [GeV].
- `showerEDep`: one row per GEANT4 **step** that deposited energy. `E` [GeV] and the step's end point: `z` from the front face, `x`, `y`, and `r` = distance from the beam axis [cm].

The beam enters at $x=y=z=0$ and travels along $z$ through a $5\times5\times10$ m box.

In [ ]:
f = uproot.open(FILE)
print(f.keys())
for name in ["edep", "showerEDep"]:
    print(f"\n{name}: {int(f[name].num_entries):,} rows")
    f[name].show()

In [ ]:
edep = f["edep"]["Edep"].array(library="np")                        # [GeV], per event
steps = f["showerEDep"].arrays(["E", "z", "r", "x", "y"], library="np")
E = steps["E"] * 1e3                                                # [MeV], per step
N = len(edep)

print(f"{N} events, {len(E):,} steps (~{len(E)//N:,} per event)")
print(f"sum of all steps / N = {E.sum()/1e3/N:.3f} GeV  vs  mean Edep = {edep.mean():.3f} GeV")

The steps have **no event number** (and the threads' output was merged), so below we look at them summed over all events.

## 2. Energy deposited per event

The Problem Set 3 plot. The dotted line is the minimum-ionizing estimate, $2.1$ MeV/cm $\times$ 10 m.

In [ ]:
counts, edges, _ = plt.hist(edep, bins=np.linspace(edep.min(), np.percentile(edep, 99.5), 60),
                            histtype="step", lw=1.5, color="#4C72B0", label=f"{E0:g} GeV $\\mu^-$")
mpv = edges[counts.argmax()] + 0.5 * np.diff(edges)[0]
plt.axvline(MIP * L / 1e3, color="gray", ls=":", label="MIP estimate")
plt.axvline(mpv, color="k", ls="--", lw=1.2, label=f"most probable: {mpv:.2f} GeV")
plt.axvline(edep.mean(), color="#DD8452", lw=1.5, label=f"mean: {edep.mean():.2f} GeV")
plt.xlabel("Energy deposited per event [GeV]")
plt.ylabel("Events")
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

A long tail to high energy (a Landau distribution): a few muons have a rare hard collision or radiate a photon, pulling the mean above the most probable value. The muon keeps most of its 10 GeV and exits the back.

## 3. Individual deposits

In [ ]:
logbins = np.logspace(np.log10(E.min()), np.log10(E.max()), 80)
plt.hist(E, bins=logbins, histtype="step", lw=1.5, color="#4C72B0", label="number of steps")
plt.hist(E, bins=logbins, weights=E / E.sum(), histtype="step", lw=1.5, ls="--",
         color="#DD8452", label="fraction of energy")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("Energy deposited in one step [MeV]")
plt.ylabel("Arbitrary")
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

Most steps deposit well under an MeV (ionization, and low-energy electrons stopping); a few big ones come from energetic secondaries.

## 4. Where along the detector?

In [ ]:
zbins = np.linspace(0, L, 101)
dEdz = np.histogram(steps["z"], bins=zbins, weights=E)[0] / N / np.diff(zbins)

plt.stairs(dEdz, zbins, color="#4C72B0", lw=1.5, label="simulation")
plt.axhline(MIP, color="gray", ls=":", label="MIP")
plt.xlabel("Depth $z$ [cm]")
plt.ylabel("$\\langle dE/dz \\rangle$ [MeV/cm]")
plt.ylim(0, 1.3 * dEdz.max())
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

Roughly flat: the muon deposits at nearly the same rate all the way through. (An electron or photon would instead shower and stop within a few metres; try it.)

## 5. How far from the axis?

In [ ]:
plt.hist2d(steps["z"], steps["r"], bins=[zbins, np.linspace(0, 20, 81)], weights=E / N,
           norm=LogNorm(), cmap="viridis")
plt.colorbar(label="Energy per event [MeV / bin]")
plt.xlabel("Depth $z$ [cm]")
plt.ylabel("Distance from axis $r$ [cm]")
plt.tight_layout()
plt.show()

The deposits drift away from the axis with depth: multiple Coulomb scattering makes the muon wander several centimetres sideways over 10 m.

## 6. The view down the beam

In [ ]:
xy = np.linspace(-30, 30, 151)
plt.figure(figsize=(5, 4.2))
plt.hist2d(steps["x"], steps["y"], bins=xy, weights=E / N, norm=LogNorm(), cmap="viridis")
plt.colorbar(label="Energy per event [MeV / bin]")
plt.xlabel("$x$ [cm]")
plt.ylabel("$y$ [cm]")
plt.gca().set_aspect("equal")
plt.tight_layout()
plt.show()

## 7. Compare runs

List the files you have (e.g. for Problem Set 3(e)); missing ones are skipped.

In [ ]:
runs = {"5 GeV": "muons_5GeV.root", "10 GeV": "muons_10GeV.root", "100 GeV": "muons_100GeV.root"}

for label, fname in runs.items():
    if not os.path.exists(fname):
        print(f"(no {fname})")
        continue
    e = uproot.open(fname)["edep"]["Edep"].array(library="np")
    plt.hist(e, bins=np.linspace(0, np.percentile(e, 99.5), 60), histtype="step", lw=1.5,
             label=f"{label}: mean {e.mean():.2f} GeV")
plt.xlabel("Energy deposited per event [GeV]")
plt.ylabel("Events")
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

**Try:** change `/gun/particle` in the macro to `e-`, `gamma`, `pi+`, or `proton`, rerun, and point `FILE` at the new output. Every plot above still works — compare the profiles.